In [ ]:
# ---- 0. 环境设置 -----------------------------------------------------------
Sys.setenv(SP_EVOLUTION_STATUS = "2")
set.seed(1)

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(edgeR)
  library(dplyr)
  library(readr)
  library(pheatmap)
  library(RColorBrewer)
  library(dendextend)
})

# ---- 1. 参数设置 -----------------------------------------------------------
date_str <- "2026-07-27_1_develop"

data_dir <- "/project/work/2026-05-25_create_summary/Data/"
plot_dir <- "/project/work/2026-05-25_create_summary/Plot/"

seurat_file <- file.path(data_dir, "2026-06-05_develop_all_human_harmony_anno.rds")
meta_file <- file.path(data_dir, "2026-06-05_develop_all_human_harmony_anno_meta.rds")

count_threshold <- 10L
min_groups_detected <- 6L
top_n <- 1000L

# Set FALSE to ignore tissue and group all tissues within each donor_age.
use_tissue <- FALSE
if (!is.logical(use_tissue) || length(use_tissue) != 1L || is.na(use_tissue)) {
  stop("use_tissue must be one TRUE/FALSE value.")
}

# Set FALSE for adult data or any analysis that should ignore stage.
use_stage <- TRUE
if (!is.logical(use_stage) || length(use_stage) != 1L || is.na(use_stage)) {
  stop("use_stage must be one TRUE/FALSE value.")
}

donor_age_order <- c(
  "70days", "79days", "87days", "91days",
  "100days", "103days", "116days",
  "137days", "141days", "142days", "162days", "165days"
)
stage_order <- c("Early", "Mid", "Late")
# Used only when use_tissue = TRUE. A single value such as "macula" is allowed.
target_tissues <- c("fovea","macula")
# target_tissues <- c("fovea","macula", "peripheral")
grouping_label <- if (use_tissue) "age_tissue" else "age"
output_prefix <- paste0(date_str, "_", grouping_label, "_similarity")
if (!use_stage) {
  output_prefix <- paste0(output_prefix, "_no_stage")
}


In [ ]:
# ---- 2. 辅助函数 -----------------------------------------------------------
# 标记线粒体、核糖体和少数高丰度技术相关基因，避免它们主导整体表达相似性。
technical_gene_flag <- function(gene_names) {
  grepl(
    "^(MT-|Mt-|mt-|MT\.|MTRNR|RPL[0-9A-Z]|RPS[0-9A-Z]|MRPL[0-9A-Z]|MRPS[0-9A-Z])",
    gene_names
  ) | gene_names %in% c("MALAT1", "XIST")
}


In [ ]:
# ---- 3. 读取数据 -----------------------------------------------------------
message("Reading Seurat object and metadata")
obj <- readRDS(seurat_file)
CONTACT_EMAIL <- readRDS(meta_file)

In [ ]:
expr_mat <- tryCatch(
  LayerData(obj, assay = "RNA", layer = "counts"),
  error = function(e) GetAssayData(obj, assay = "RNA", slot = "counts")
)

metadata <- CONTACT_EMAIL
metadata$donor_age <- trimws(as.character(metadata$donor_age))
if (use_stage) {
  metadata$stage <- trimws(as.character(metadata$stage))
}
if (use_tissue) {
  metadata$tissue <- trimws(as.character(metadata$tissue))
}

common_cells <- intersect(colnames(expr_mat), rownames(metadata))
expr_mat <- expr_mat[, common_cells, drop = FALSE]
metadata <- metadata[common_cells, , drop = FALSE]
metadata <- metadata[colnames(expr_mat), , drop = FALSE]

if (use_tissue) {
  if (
    length(target_tissues) == 0L ||
    anyDuplicated(target_tissues) ||
    !all(target_tissues %in% c("fovea", "macula", "peripheral"))
  ) {
    stop("target_tissues can only contain fovea, macula, and/or peripheral.")
  }
  keep_cells <- metadata$donor_age %in% donor_age_order &
    metadata$tissue %in% target_tissues
} else {
  metadata$tissue <- "all_tissues"
  keep_cells <- metadata$donor_age %in% donor_age_order
}
if (use_stage) {
  keep_cells <- keep_cells & metadata$stage %in% stage_order
}

expr_mat <- expr_mat[, keep_cells, drop = FALSE]
metadata <- metadata[keep_cells, , drop = FALSE]
if (ncol(expr_mat) == 0L) {
  stop("No cells remain after donor_age, stage, and tissue filtering.")
}

In [ ]:
# ---- 4. 构建 age_tissue pseudobulk ----------------------------------------
message(
  "Building raw-count pseudobulk by donor_age",
  if (use_tissue) " x tissue" else ""
)
if (use_tissue) {
  metadata$sample_id <- paste(
    metadata$donor_age, metadata$tissue, sep = "__"
  )
  sample_levels <- as.vector(t(outer(
    donor_age_order, target_tissues, paste, sep = "__"
  )))
} else {
  metadata$sample_id <- metadata$donor_age
  sample_levels <- donor_age_order
}
sample_levels <- sample_levels[sample_levels %in% metadata$sample_id]

sample_factor <- factor(metadata$sample_id, levels = sample_levels)
group_design <- sparseMatrix(
  i = seq_along(sample_factor),
  j = as.integer(sample_factor),
  x = 1,
  dims = c(length(sample_factor), length(levels(sample_factor))),
  dimnames = list(colnames(expr_mat), levels(sample_factor))
)

pseudobulk_counts <- as(expr_mat %*% group_design, "dgCMatrix")

if (use_stage) {
  age_stage <- metadata %>%
    distinct(donor_age, stage) %>%
    arrange(match(donor_age, donor_age_order))
  age_to_stage <- setNames(age_stage$stage, age_stage$donor_age)
} else {
  age_to_stage <- setNames(
    rep("not_applicable", length(donor_age_order)),
    donor_age_order
  )
}

cell_counts <- as.integer(table(metadata$sample_id)[colnames(pseudobulk_counts)])
sample_info <- data.frame(
  sample_id = colnames(pseudobulk_counts),
  donor_age = if (use_tissue) {
    sub("__.*$", "", colnames(pseudobulk_counts))
  } else {
    colnames(pseudobulk_counts)
  },
  tissue = if (use_tissue) {
    sub("^.*__", "", colnames(pseudobulk_counts))
  } else {
    "all_tissues"
  },
  n_cells = cell_counts,
  library_size = as.numeric(Matrix::colSums(pseudobulk_counts)),
  detected_genes = as.numeric(Matrix::colSums(pseudobulk_counts > 0)),
  stringsAsFactors = FALSE
)
sample_info$stage <- unname(age_to_stage[sample_info$donor_age])
sample_info <- sample_info[, c(
  "sample_id",  "donor_age", "stage", "tissue",
  "n_cells", "library_size", "detected_genes"
)]

In [ ]:
# ---- 5. 基因过滤和标准化 ----------------------------------------------------
message("Filtering genes and normalizing with edgeR TMM + logCPM")
detected_groups <- Matrix::rowSums(pseudobulk_counts >= count_threshold)
keep_genes <- detected_groups >= min_groups_detected

filtered_counts <- as.matrix(pseudobulk_counts[keep_genes, , drop = FALSE])
dge <- DGEList(counts = filtered_counts)
dge <- calcNormFactors(dge, method = "TMM")
logcpm <- cpm(
  dge,
  log = TRUE,
  prior.count = 1,
  normalized.lib.sizes = TRUE
)

In [ ]:
# ---- 6. 选择高变基因并计算相似性 -------------------------------------------
message("Selecting top variable non-technical genes")
nontechnical <- !technical_gene_flag(rownames(logcpm))
logcpm_nontechnical <- logcpm[nontechnical, , drop = FALSE]

gene_var <- apply(logcpm_nontechnical, 1, var, na.rm = TRUE)
eligible <- is.finite(gene_var) & gene_var > 0
ranked_genes <- names(sort(gene_var[eligible], decreasing = TRUE))
selected_genes <- ranked_genes[seq_len(min(top_n, length(ranked_genes)))]

similarity_mat_input <- logcpm_nontechnical[selected_genes, , drop = FALSE]
gene_sd <- apply(similarity_mat_input, 1, sd, na.rm = TRUE)
similarity_mat_input <- similarity_mat_input[
  is.finite(gene_sd) & gene_sd > 0,
  ,
  drop = FALSE
]
selected_genes <- rownames(similarity_mat_input)

message("Computing Pearson correlation and hierarchical clustering")
cor_mat <- cor(
  similarity_mat_input,
  method = "pearson",
  use = "pairwise.complete.obs"
)
diag(cor_mat) <- 1

if (any(!is.finite(cor_mat))) {
  stop("Correlation matrix contains non-finite values.")
}

dist_mat <- as.dist(1 - cor_mat)
hc <- hclust(dist_mat, method = "average")

In [ ]:
# ---- 手动旋转 dendrogram，仅调整叶节点显示顺序，不改变聚类结构 ----
dend <- as.dendrogram(hc)

target_order <- c(
  "165days",
  "137days",
  "142days",  
  "141days",
  "162days",
  "116days",
  "103days",
  "100days",
  "87days",
  "91days",
  "70days",
  "79days"
)

dend <- rotate(dend, order = target_order)

hc <- as.hclust(dend)


In [ ]:
# ---- 8. 绘制相似性热图 ------------------------------------------------------
# 去掉 tissue 与 n_cells 轨道 (age-only merge 里 tissue 恒为 all_tissues; n_cells 不需要)
annotation <- sample_info[, c("stage", "donor_age"), drop = FALSE]
rownames(annotation) <- sample_info$sample_id
annotation <- annotation[colnames(cor_mat), , drop = FALSE]
annotation$stage <- factor(annotation$stage, levels = c(stage_order, "not_applicable"))
annotation$donor_age <- factor(annotation$donor_age, levels = donor_age_order)

cols.palette1 <- c(
  brewer.pal(9, "Set1"), brewer.pal(8, "Dark2"),
  brewer.pal(8, "Set2"), brewer.pal(8, "Accent")
)
cols.age <- c(cols.palette1[18:29], cols.palette1[6], cols.palette1[1:5], cols.palette1[8])
names(cols.age) <- c(
  "70days", "79days", "87days", "91days", "100days", "103days",
  "116days", "137days", "141days", "142days", "162days", "165days",
  "21years", "25years", "67years", "70years", "75years", "86years"
)
annotation_colors <- list(
  stage = c(Early = "#59A14F", Mid = "#EDC948", Late = "#B07AA1",
            not_applicable = "#BDBDBD")[levels(droplevels(annotation$stage))],
  donor_age = cols.age[levels(droplevels(annotation$donor_age))]
)

plot_cor_mat <- cor_mat
rownames(plot_cor_mat) <- gsub("__", "_", rownames(plot_cor_mat))
colnames(plot_cor_mat) <- gsub("__", "_", colnames(plot_cor_mat))
plot_annotation <- annotation
rownames(plot_annotation) <- gsub("__", "_", rownames(plot_annotation))

heat_colors <- colorRampPalette(c("#313695", "#74ADD1", "#FFFFBF", "#F46D43", "#A50026"))(100)
heat_breaks <- seq(min(cor_mat), max(cor_mat), length.out = length(heat_colors) + 1)

In [ ]:
# ---- 8b. 绘制相似性热图 (样式对齐 adult reordered 图: cell20 / 字体12,11 / 无标题 / treeheight40) ----
library(grid)

ph <- pheatmap(
  plot_cor_mat,
  color = heat_colors,
  breaks = heat_breaks,
  cluster_rows = hc,
  cluster_cols = hc,
  clustering_distance_rows = dist_mat,
  clustering_distance_cols = dist_mat,
  annotation_row = plot_annotation,
  annotation_col = plot_annotation,
  annotation_colors = annotation_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  cellwidth = 20,
  cellheight = 20,
  fontsize = 12,
  fontsize_row = 11,
  fontsize_col = 11,
  angle_col = 90,
  border_color = NA,
  treeheight_row = 40,
  treeheight_col = 40,
  main = NA,
  silent = TRUE
)

# 图例垂直居中: 在 legend cell 视口内实测各图例范围; 图例过高则顶部对齐防裁切, 否则居中。
center_legends <- function(ph, fig_h, top_pad = 12) {
  g <- ph$gtable
  fig_h_pt <- fig_h * 72
  il <- which(g$layout$name == "legend")
  rowH <- vapply(seq_along(g$heights),
                 function(k) convertHeight(g$heights[k], "bigpts", valueOnly = TRUE), numeric(1))
  t_row <- g$layout$t[il]; b_row <- g$layout$b[il]
  top_margin <- if (t_row > 1) sum(rowH[1:(t_row - 1)]) else 0
  cell_h <- sum(rowH[t_row:b_row])
  cell_ymid <- fig_h_pt - top_margin - cell_h / 2
  cell_bottom <- cell_ymid - cell_h / 2
  pushViewport(viewport(y = unit(cell_ymid, "bigpts"),
                        height = unit(cell_h, "bigpts"), just = "centre"))
  for (nm in c("legend", "annotation_legend")) {
    i <- which(g$layout$name == nm); if (length(i) != 1) next
    gr <- g$grobs[[i]]
    ys <- unlist(lapply(gr$children,
      function(ch) if (!is.null(ch$y)) convertY(ch$y, "bigpts", valueOnly = TRUE)))
    leg_h <- max(ys) - min(ys)
    if (leg_h > fig_h_pt - 2 * top_pad) {          # 过高: 顶部对齐
      delta <- (fig_h_pt - top_pad) - (cell_bottom + max(ys))
    } else {                                       # 否则: 垂直居中
      delta <- fig_h_pt / 2 - (cell_bottom + (min(ys) + max(ys)) / 2)
    }
    gr$children <- lapply(gr$children,
      function(ch) { if (!is.null(ch$y)) ch$y <- ch$y + unit(delta, "bigpts"); ch })
    g$grobs[[i]] <- gr; g$layout$clip[i] <- "off"
  }
  popViewport()
  g
}

In [ ]:
# 输出 age 表达相似性 PDF 热图 (图例垂直居中; 尺寸/字体与 adult reordered 图一致)。
heatmap_pdf <- file.path(plot_dir, paste0(output_prefix, "_heatmap_merge.pdf"))

fig_w <- 9; fig_h <- 8.5
pdf(heatmap_pdf, width = fig_w, height = fig_h, onefile = FALSE)
grid.newpage()
grid.draw(center_legends(ph, fig_h))
dev.off()

message("Done")
message(
  "tissue grouping = ",
  if (use_tissue) paste(target_tissues, collapse = ", ") else "disabled; all tissues combined"
)
message("stage information = ", if (use_stage) "used" else "ignored")
message("pseudobulk samples = ", ncol(pseudobulk_counts))
message("filtered genes = ", sum(keep_genes))
message("selected genes = ", length(selected_genes))